In [1]:
# ===============================
# 1. Libraries
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score


In [2]:
# ===============================
# 2. Load Dataset
# ===============================
data = pd.read_csv("FD001_combined.csv")

# basic checks
print(data.isnull().sum())


unit_number       0
time_in_cycles    0
OS1               0
OS2               0
OS3               0
sensor_1          0
sensor_2          0
sensor_3          0
sensor_4          0
sensor_5          0
sensor_6          0
sensor_7          0
sensor_8          0
sensor_9          0
sensor_10         0
sensor_11         0
sensor_12         0
sensor_13         0
sensor_14         0
sensor_15         0
sensor_16         0
sensor_17         0
sensor_18         0
sensor_19         0
sensor_20         0
sensor_21         0
RUL               0
dtype: int64


In [3]:
# ===============================
# 3. Central Tendency (EDA)
# ===============================
central = pd.DataFrame(index=['mean','median','mode'], columns=data.columns)

for col in data.columns:
    central.loc['mean', col]   = data[col].mean()
    central.loc['median', col] = data[col].median()
    central.loc['mode', col]   = data[col].mode()[0]

central


,unit_number,time_in_cycles,OS1,OS2,OS3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
mean,51.521066,96.393572,-0.00001,0.000003,100.0,518.67,642.601005,1589.581927,1407.303559,14.62,...,2388.086395,8141.887005,8.435816,0.03,392.962523,2388.0,100.0,38.845871,23.307581,120.788775
median,52.0,88.0,-0.0,-0.0,100.0,518.67,642.56,1589.21,1406.45,14.62,...,2388.08,8139.62,8.4324,0.03,393.0,2388.0,100.0,38.86,23.3166,121.0
mode,49,1,0.0,-0.0003,100.0,518.67,642.49,1589.76,1401.27,14.62,...,2388.07,8138.31,8.4309,0.03,393,2388,100.0,38.89,23.3222,137


In [4]:
# ===============================
# 4. Features & Target
# ===============================
X = data[['unit_number', 'time_in_cycles', 'OS1', 'OS2', 'OS3',
          'sensor_1','sensor_2','sensor_3','sensor_4','sensor_5',
          'sensor_6','sensor_7','sensor_8','sensor_9','sensor_10',
          'sensor_11','sensor_12','sensor_13','sensor_14','sensor_15',
          'sensor_16','sensor_17','sensor_18','sensor_19','sensor_20',
          'sensor_21']]

y = data['RUL']


In [5]:
# ===============================
# 5. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0
)


In [6]:
# ===============================
# 6. Baseline Linear Regression
# ===============================
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print("Linear Regression R2:", r2_score(y_test, y_pred))


Linear Regression R2: 0.6313738758118173


In [7]:
# ===============================
# 7. Multiple Models Comparison
# ===============================
models = [
    ("Linear", LinearRegression()),
    ("Ridge", Ridge()),
    ("Lasso", Lasso()),
    ("SVR", SVR()),
    ("Decision Tree", DecisionTreeRegressor(random_state=0)),
    ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=0)),
    ("KNN", KNeighborsRegressor(n_neighbors=5))
]

for name, model in models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} R2 :", r2_score(y_test, y_pred))


Linear R2 : 0.6313738758118173
Ridge R2 : 0.6312942800149289
Lasso R2 : 0.6223344062937715
SVR R2 : 0.02108096413991889
Decision Tree R2 : 0.6876374995428916
Random Forest R2 : 0.8334281379209395
KNN R2 : 0.7237259939072831


In [37]:
# ===============================
# 8. Feature Selection (SelectKBest)
# ===============================
kbest = SelectKBest(score_func=f_regression, k=20)
X_kbest = kbest.fit_transform(X, y)

# selected feature names
selected_features = X.columns[kbest.get_support()]
print("Selected Features:", selected_features)


Selected Features: Index(['unit_number', 'time_in_cycles', 'OS1', 'OS2', 'sensor_2', 'sensor_3',
       'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9',
       'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
       'sensor_17', 'sensor_20', 'sensor_21'],
      dtype='object')


C:\Users\anand\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:381: RuntimeWarning: invalid value encountered in sqrt
  X_norms = np.sqrt(row_norms(X.T, squared=True) - n_samples * X_means**2)


In [38]:
# ===============================
# 9. Split + Scaling Function
# ===============================
def split_scalar(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(
        indep_X, dep_Y, test_size=0.25, random_state=0
    )
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test  = sc.transform(X_test)
    return X_train, X_test, y_train, y_test


In [39]:
# ===============================
# 10. R2 Prediction Function
# ===============================
def r2_prediction(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return r2_score(y_test, y_pred)


In [40]:
# ===============================
# 11. Model Functions
# ===============================
def Linear_Model(X_train, y_train, X_test, y_test):
    model = LinearRegression()
    model.fit(X_train, y_train)
    return r2_prediction(model, X_test, y_test)

def SVM_Linear(X_train, y_train, X_test, y_test):
    model = SVR(kernel='linear')
    model.fit(X_train, y_train)
    return r2_prediction(model, X_test, y_test)

def SVM_RBF(X_train, y_train, X_test, y_test):
    model = SVR(kernel='rbf')
    model.fit(X_train, y_train)
    return r2_prediction(model, X_test, y_test)

def Decision_Model(X_train, y_train, X_test, y_test):
    model = DecisionTreeRegressor(random_state=0)
    model.fit(X_train, y_train)
    return r2_prediction(model, X_test, y_test)

def Random_Model(X_train, y_train, X_test, y_test):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    return r2_prediction(model, X_test, y_test)


In [41]:
# ===============================
# 12. Apply Models on KBest Features
# ===============================
X_train_k, X_test_k, y_train_k, y_test_k = split_scalar(X_kbest, y)

results = {
    "Linear": Linear_Model(X_train_k, y_train_k, X_test_k, y_test_k),
    "SVM Linear": SVM_Linear(X_train_k, y_train_k, X_test_k, y_test_k),
    "SVM RBF": SVM_RBF(X_train_k, y_train_k, X_test_k, y_test_k),
    "Decision Tree": Decision_Model(X_train_k, y_train_k, X_test_k, y_test_k),
    "Random Forest": Random_Model(X_train_k, y_train_k, X_test_k, y_test_k)
}

result_df = pd.DataFrame(results, index=["R2 Score"])
result_df


,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.631968,0.617943,0.664714,0.670712,0.838069


In [16]:
result_df # 2

,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.612486,0.598825,0.642734,0.282098,0.640923


In [21]:
result_df #3

,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.612486,0.598825,0.642734,0.282098,0.640923


In [30]:
result_df #10

,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.617829,0.604189,0.644313,0.299251,0.657264


In [36]:
result_df # 15

,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.625241,0.611832,0.663981,0.353204,0.685683


In [42]:
result_df

,Linear,SVM Linear,SVM RBF,Decision Tree,Random Forest
R2 Score,0.631968,0.617943,0.664714,0.670712,0.838069
